# Approach 2 · Additive experiments on the latest Baseline 2

Reference [Baseline 2 on GitHub master](https://github.com/COS30082-ML-Project-2026/COS30082-Plant-Disease-OOD/blob/0fece2daff4917431bb2a5b16b65718ab100d05c/baselines/baseline2_multitask.ipynb) and its [published release](https://github.com/COS30082-ML-Project-2026/COS30082-Plant-Disease-OOD/releases/tag/baseline2-dinov2-base-frozen-v1). The release reports 63.90% seen and 9.60% unseen Top-1. Use this notebook's B0 rerun as the matched control.

The experiment tests whether the loss and augmentation additions improve generalization while preserving the original 768-dimensional classifier input and fusion rule.

## 1. Setup

Open in the team repository with the project's Jupyter environment. `DEBUG=True` uses the Baseline 2 debug data subsamples and one epoch. `DEBUG=False` runs the complete matched ablation and writes checkpoints and official metrics. The first run needs the same pretrained DINOv2 weights used by Baseline 2.

In [ ]:
DEBUG = True

from pathlib import Path
import copy
import hashlib
import json
import sys

import pandas as pd
import timm
import torch
from IPython.display import display
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF

here = Path.cwd()
ROOT = next((p for p in (here, *here.parents) if (p / "common" / "dataset.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook inside the team repository.")
sys.path.insert(0, str(ROOT))

from common import config as team
from common.dataset import PlantDataset, get_loaders, get_pairs, get_transforms, load_split
from common.evaluate import PERTURBATIONS, harmonic_mean, save_confusion, top_k_accuracy

BACKBONE = "vit_base_patch14_dinov2.lvd142m"
REFERENCE_COMMIT = "0fece2daff4917431bb2a5b16b65718ab100d05c"
BATCH_SIZE = 8 if DEBUG else 16
EPOCHS = 1 if DEBUG else 20
DEVICE = team.DEVICE
NUM_CROPS, NUM_DISEASES = team.NUM_CROPS, team.NUM_DISEASES
team.NUM_WORKERS = 0
RUN_NAME = "approach2_baseline2_matched" + ("_debug" if DEBUG else "")
OUTPUT_DIR = team.RESULTS_DIR / RUN_NAME
DEVELOPMENT_DIR = OUTPUT_DIR / "development"
FINAL_DIR = OUTPUT_DIR / "final"
OFFICIAL_DIR = OUTPUT_DIR / "official"
CHECKPOINT_DIR = team.CHECKPOINT_DIR / RUN_NAME
FULL_CHECKPOINT = CHECKPOINT_DIR / "model.pt"
ABLATION_HEADS = CHECKPOINT_DIR / "comparison_heads.pt"
LOCK = FINAL_DIR / "selection_lock.json"

LR, WEIGHT_DECAY, SMOOTHING = 1e-4, 1e-4, 0.1
DECOR_WEIGHT, SUPCON_WEIGHT, TEMPERATURE = 0.05, 0.1, 0.1
BLUR_PROB, BLUR_SIGMA = 0.15, 0.7
GAMMAS = [step / 2 for step in range(25)]
RECIPES = {
    "B0_Baseline2": (0.0, 0.0, False),
    "A1_Decorrelation": (DECOR_WEIGHT, 0.0, False),
    "A2_Add_SupCon": (DECOR_WEIGHT, SUPCON_WEIGHT, False),
    "A3_Add_Blur": (DECOR_WEIGHT, SUPCON_WEIGHT, True),
}
BASELINE, FINAL = next(iter(RECIPES)), next(reversed(RECIPES))

if not team.TRAIN_CSV.is_file() or not team.IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Missing training metadata or images under {team.DATA_DIR}.")
if team.NUM_PAIRS != team.NUM_SEEN_PAIRS + team.NUM_UNSEEN_PAIRS:
    raise ValueError("Use the team's 323/333 common files.")
print(f"{DEVICE} | FP32 | batch {BATCH_SIZE} | {EPOCHS} epochs | DEBUG={DEBUG}")

## 2. Data

Every main recipe uses Baseline 2's train/seen-validation split, seed 42, ImageNet normalization, random crop, flip, rotation, color jitter and shuffled batch size 16. The separate shadow heads omit ten training-derived combinations from their fit data for calibration only. Five proxy pairs and seen validation images choose gamma. Another five proxy pairs and disjoint seen images check it. Main recipe training and model selection still use Baseline 2's original split and criterion.

In [ ]:
def image_key(value):
    result = Path(str(value).strip().replace("\\", "/"))
    if not result.name or result.is_absolute() or ".." in result.parts or ":" in str(result):
        raise ValueError(f"Expected a relative image filename, got {value!r}.")
    return result.as_posix()


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path, value):
    temp = path.with_suffix(".json.tmp")
    temp.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temp.replace(path)


def atomic_save(path, value):
    temp = path.with_suffix(".pt.tmp")
    torch.save(value, temp)
    temp.replace(path)


train_full = load_split(team.TRAIN_CSV)
train_full["image"] = train_full.image.map(image_key)
if not train_full.image.is_unique:
    raise ValueError("Duplicate training images.")
train_pairs = get_pairs()
if train_pairs.shape != (team.NUM_SEEN_PAIRS, 2):
    raise ValueError("Unexpected training pair table.")
_, image_transform = get_transforms("imagenet")


def calibration_split(fit_frame, val_frame):
    """Hold out complete training pairs from the shadow heads only."""
    grouped = fit_frame.groupby("pair_row", sort=True)
    rows = grouped.agg(n=("image", "size"), crop=("crop", "first"), disease=("disease", "first"))
    crops = rows.crop.value_counts().to_dict()
    diseases = rows.disease.value_counts().to_dict()
    eligible = rows[rows.n.ge(5)].sample(frac=1, random_state=team.SEED + 1)
    held = []
    for pair_row, row in eligible.iterrows():
        if crops[row.crop] > 1 and diseases[row.disease] > 1:
            held.append(int(pair_row))
            crops[row.crop] -= 1
            diseases[row.disease] -= 1
        if len(held) == 10:
            break
    if len(held) < 10:
        raise ValueError("Ten train-derived holdouts with fit support were not available.")
    seen = val_frame.loc[~val_frame.pair_row.isin(held)].sample(frac=1, random_state=team.SEED + 2)
    cut = len(seen) // 2
    frames = {
        "tune_seen": seen.iloc[:cut],
        "tune_proxy": train_full.loc[train_full.pair_row.isin(held[:5])],
        "check_seen": seen.iloc[cut:],
        "check_proxy": train_full.loc[train_full.pair_row.isin(held[5:])],
    }
    if any(group.empty for group in frames.values()):
        raise ValueError("Calibration needs both seen and proxy images in each group.")
    fit_mask = torch.ones(len(train_pairs), dtype=torch.bool)
    fit_mask[held] = False
    return fit_mask, frames


settings = {
    "reference_commit": REFERENCE_COMMIT, "backbone": BACKBONE, "seed": team.SEED,
    "image_size": team.IMAGE_SIZE, "resize_size": team.RESIZE_SIZE, "val_fraction": team.VAL_FRACTION,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "lr": LR, "weight_decay": WEIGHT_DECAY,
    "optimizer": "AdamW", "schedule": "CosineAnnealingLR", "label_smoothing": SMOOTHING,
    "checkpoint_rule": "best seen Top-1 on baseline validation",
    "recipes": {name: list(values) for name, values in RECIPES.items()},
    "temperature": TEMPERATURE, "blur_probability": BLUR_PROB, "blur_sigma": BLUR_SIGMA,
    "calibration_grid": GAMMAS, "train_csv_sha256": sha256_file(team.TRAIN_CSV),
    "torch": str(torch.__version__), "timm": timm.__version__, "debug": DEBUG,
}

## 3. Model and losses

All four recipes use one frozen DINOv2 ViT-B/14 with the original two Linear Heads. The raw score stays `log P(crop) + log P(disease)`. Each recipe adds one controlled change.

| Recipe | Addition compared with the preceding recipe |
|---|---|
| B0_Baseline2 | GitHub Baseline 2: two cross-entropy losses with label smoothing 0.1 |
| A1_Decorrelation | Weight 0.05 on cross-correlation of centered crop and disease logits |
| A2_Add_SupCon | Weight 0.10 on disease-logit SupCon, temperature 0.10, same disease and different crop positives |
| A3_Add_Blur | Apply mild Gaussian blur to 15% of original training-augmented images |

The auxiliary losses use trainable outputs from the existing heads. Their gradients reach the same original classifiers, and their logits continue to drive prediction. All recipes have identical architecture and parameter count. SupCon uses positives found in the original minibatches, and the logs give the usable anchor fraction. Gaussian blur is the new augmentation because Baseline 2 already uses random flips. It uses a separate random stream.

In [ ]:
class MultiTaskNet(nn.Module):
    """Latest Baseline 2 architecture."""
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0,
                                          img_size=team.IMAGE_SIZE)
        self.backbone.requires_grad_(False)
        self.crop_head = nn.Linear(self.backbone.num_features, NUM_CROPS)
        self.disease_head = nn.Linear(self.backbone.num_features, NUM_DISEASES)

    def forward(self, images):
        features = self.backbone(images)
        return self.crop_head(features), self.disease_head(features)


def pair_scores(crop_logits, disease_logits, pairs):
    pairs = pairs.to(crop_logits.device)
    crop_log = F.log_softmax(crop_logits.float(), dim=1)
    disease_log = F.log_softmax(disease_logits.float(), dim=1)
    return crop_log[:, pairs[:, 0]] + disease_log[:, pairs[:, 1]]


def center_logits(logits):
    # Remove the per-image offset to which categorical predictions are invariant.
    return logits - logits.mean(dim=1, keepdim=True)


def decorrelation(crop, disease):
    def standardize(x):
        x = x - x.mean(0)
        return x / (x.square().mean(0) + 1e-4).sqrt()

    cross_correlation = standardize(crop).T @ standardize(disease) / len(crop)
    return cross_correlation.square().mean()


def cross_crop_supcon(embedding, crop, disease):
    z = F.normalize(embedding, dim=1)
    same_disease = disease[:, None].eq(disease[None, :])
    positives = same_disease & crop[:, None].ne(crop[None, :])
    negatives = ~same_disease
    similarity = z @ z.T / TEMPERATURE
    normalizer = similarity.masked_fill(~(positives | negatives), -1e9).logsumexp(dim=1)
    counts = positives.sum(dim=1)
    positive_mean = similarity.masked_fill(~positives, 0.0).sum(dim=1) / counts.clamp_min(1)
    usable = (counts > 0) & negatives.any(dim=1)
    per_anchor = (normalizer - positive_mean).masked_fill(~usable, 0.0)
    return per_anchor.sum() / usable.sum().clamp_min(1), usable.sum()

## 4. Training and checkpoint selection

Train all main recipes on identical sampled batches and initialization. The frozen backbone runs once per base view, with an extra forward for the small blurred subset. Use FP32, AdamW at 1e-4, weight decay 1e-4 and the Baseline 2 cosine schedule for 20 epochs. Each recipe saves its highest seen-validation Top-1 checkpoint, with ties going to the earlier epoch. A3 is the predesignated full Approach 2. The shadow calibration heads share feature passes but receive gradients only from their own fit images. Each gamma comes from the corresponding shadow epoch, is fixed before evaluation, and is accompanied by held-out check metrics.

In [ ]:
def trainable_heads(model):
    return nn.ModuleDict({"crop_head": copy.deepcopy(model.crop_head),
                          "disease_head": copy.deepcopy(model.disease_head)})


def forward_heads(heads, features):
    return heads["crop_head"](features), heads["disease_head"](features)


def snapshot(heads):
    return {name: value.detach().cpu().clone() for name, value in heads.state_dict().items()}


def add_blur_view(backbone, images, features, generator):
    selected = torch.rand(len(images), generator=generator) < BLUR_PROB
    if not selected.any():
        return features
    selected = selected.to(images.device)
    blurred = TF.gaussian_blur(images[selected], kernel_size=[5, 5], sigma=[BLUR_SIGMA, BLUR_SIGMA])
    with torch.no_grad():
        augmented = features.clone()
        augmented[selected] = backbone(blurred)
    return augmented


def train_step(heads, optimizer, features, crop, disease, recipe, criterion):
    decor_weight, con_weight, _ = RECIPES[recipe]
    crop_logits, disease_logits = forward_heads(heads, features)
    loss = criterion(crop_logits, crop) + criterion(disease_logits, disease)
    anchors = 0
    if decor_weight:
        loss = loss + decor_weight * decorrelation(center_logits(crop_logits), center_logits(disease_logits))
    if con_weight:
        contrast, usable = cross_crop_supcon(center_logits(disease_logits), crop, disease)
        loss = loss + con_weight * contrast
        anchors = int(usable.detach())
    if not torch.isfinite(loss):
        raise FloatingPointError(f"Non-finite loss in {recipe}.")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return loss.item(), anchors


@torch.no_grad()
def score_loader(backbone, heads_by_name, loader):
    scores = {name: [] for name in heads_by_name}
    labels = []
    for images, _, _, target in loader:
        features = backbone(images.to(DEVICE))
        for name, heads in heads_by_name.items():
            heads.eval()
            scores[name].append(pair_scores(*forward_heads(heads, features), train_pairs).cpu())
        labels.append(target.cpu())
    return {name: torch.cat(values) for name, values in scores.items()}, torch.cat(labels)


def calibration_metrics(seen, proxy, fit_mask, gamma):
    seen_scores, seen_y = seen
    proxy_scores, proxy_y = proxy
    penalty = gamma * fit_mask
    s = top_k_accuracy(seen_scores - penalty, seen_y, 1)
    u = top_k_accuracy(proxy_scores - penalty, proxy_y, 1)
    return {"gamma": gamma, "seen_top1": s, "proxy_top1": u, "harmonic_top1": harmonic_mean(s, u)}


def choose_calibration(backbone, heads, frames, fit_mask):
    groups = {}
    for name, frame in frames.items():
        loader = DataLoader(PlantDataset(frame, image_transform, "imagenet"), batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=0)
        groups[name] = score_loader(backbone, heads, loader)

    def pair_for(recipe, name):
        scores, labels = groups[name]
        return scores[recipe], labels

    frozen, trials, checks = {}, [], []
    for recipe in heads:
        tune = [pair_for(recipe, n) for n in ("tune_seen", "tune_proxy")]
        check = [pair_for(recipe, n) for n in ("check_seen", "check_proxy")]
        grid = [calibration_metrics(*tune, fit_mask, gamma) for gamma in GAMMAS]
        best = max(grid, key=lambda row: (row["harmonic_top1"], row["proxy_top1"],
                                         row["seen_top1"], -row["gamma"]))
        frozen[recipe] = best["gamma"]
        trials.extend({"recipe": recipe, **row} for row in grid)
        for scoring, gamma in (("raw", 0.0), ("locked_calibrated", best["gamma"])):
            checks.append({"recipe": recipe, "scoring": scoring,
                           **calibration_metrics(*check, fit_mask, gamma)})
    pd.DataFrame(trials).to_csv(DEVELOPMENT_DIR / "calibration_tune_grid.csv", index=False)
    pd.DataFrame(checks).to_csv(DEVELOPMENT_DIR / "calibration_check.csv", index=False)
    return frozen


def read_final_lock():
    if not LOCK.is_file() or not FULL_CHECKPOINT.is_file() or not ABLATION_HEADS.is_file():
        raise FileNotFoundError("The final selection lock and both checkpoint files must exist.")
    saved = json.loads(LOCK.read_text(encoding="utf-8"))
    if (saved["settings"] != settings
            or saved["full_checkpoint_sha256"] != sha256_file(FULL_CHECKPOINT)
            or saved["ablation_heads_sha256"] != sha256_file(ABLATION_HEADS)):
        raise RuntimeError("Run settings or checkpoint hashes changed. Archive this run before starting another.")
    return saved


def train_and_save():
    if LOCK.is_file():
        saved = read_final_lock()
        print(f"Reusing completed run at {OUTPUT_DIR}")
        return saved

    DEVELOPMENT_DIR.mkdir(parents=True, exist_ok=True)
    team.set_seed()
    train_loader, val_loader, seen, unseen = get_loaders(BATCH_SIZE, norm="imagenet", debug=DEBUG)
    if seen is not None or unseen is not None:
        raise RuntimeError("Training loaders must never return official test data.")
    train_loader.pin_memory = val_loader.pin_memory = False
    fit_frame, val_frame = train_loader.dataset.frame, val_loader.dataset.frame
    fit_mask, calibration_frames = (None, {}) if DEBUG else calibration_split(fit_frame, val_frame)

    model = MultiTaskNet(pretrained=True).to(DEVICE).eval()
    main = {recipe: trainable_heads(model) for recipe in RECIPES}
    shadow = {} if DEBUG else {recipe: trainable_heads(model) for recipe in (BASELINE, FINAL)}
    units = {(stage, name): heads for stage, suite in (("main", main), ("shadow", shadow))
             for name, heads in suite.items()}
    optimizers = {key: torch.optim.AdamW(heads.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
                  for key, heads in units.items()}
    schedulers = {key: torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
                  for key, opt in optimizers.items()}
    criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)
    blur_rng = torch.Generator().manual_seed(team.SEED + 172)
    fit_device = fit_mask.to(DEVICE) if fit_mask is not None else None
    history, best = [], {}
    for epoch in range(1, EPOCHS + 1):
        model.backbone.eval()
        for heads in units.values():
            heads.train()
        totals = {name: [0.0, 0] for name in RECIPES}
        examples = 0
        for step, (images, crop, disease, pair_rows) in enumerate(train_loader, 1):
            images, crop, disease = images.to(DEVICE), crop.to(DEVICE), disease.to(DEVICE)
            with torch.no_grad():
                clean = model.backbone(images)
            blurred = add_blur_view(model.backbone, images, clean, blur_rng)
            keep = fit_device[pair_rows.to(DEVICE)] if fit_device is not None else None
            for (stage, recipe), heads in units.items():
                features = blurred if RECIPES[recipe][2] else clean
                if stage == "shadow":
                    if keep.any():
                        train_step(heads, optimizers[(stage, recipe)], features[keep], crop[keep],
                                   disease[keep], recipe, criterion)
                else:
                    loss, anchors = train_step(heads, optimizers[(stage, recipe)], features, crop,
                                               disease, recipe, criterion)
                    totals[recipe][0] += loss * len(images)
                    totals[recipe][1] += anchors
            examples += len(images)
            if step == 1 or step % 50 == 0 or step == len(train_loader):
                print(f"Epoch {epoch}/{EPOCHS} · batch {step}/{len(train_loader)}", flush=True)
        for scheduler in schedulers.values():
            scheduler.step()
        validation, labels = score_loader(model.backbone, main, val_loader)
        for recipe, scores in validation.items():
            top1 = top_k_accuracy(scores, labels, 1)
            row = {"epoch": epoch, "recipe": recipe, "val_seen_top1": top1,
                   "train_loss": totals[recipe][0] / examples,
                   "supcon_anchor_fraction": totals[recipe][1] / examples}
            history.append(row)
            print(f"{recipe} · epoch {epoch} · seen validation Top-1 {top1:.2f}%")
            if recipe not in best or top1 > best[recipe]["top1"]:
                best[recipe] = {"epoch": epoch, "top1": top1, "weights": snapshot(main[recipe]),
                                "shadow": snapshot(shadow[recipe]) if recipe in shadow else None}

    pd.DataFrame(history).to_csv(DEVELOPMENT_DIR / "training_history.csv", index=False)
    if not DEBUG and not any(row["recipe"] == FINAL and row["supcon_anchor_fraction"] > 0 for row in history):
        raise RuntimeError("No usable cross-crop SupCon anchors in the unchanged Baseline 2 batches.")
    if DEBUG:
        return {"selection": {name: {"epoch": result["epoch"], "seen_top1": result["top1"]}
                             for name, result in best.items()}}
    for recipe, heads in shadow.items():
        heads.load_state_dict(best[recipe]["shadow"])
    gamma = choose_calibration(model.backbone, shadow, calibration_frames, fit_mask)

    selection = {name: {"epoch": result["epoch"], "seen_top1": result["top1"]}
                 for name, result in best.items()}
    final_heads = main[FINAL]
    final_heads.load_state_dict(best[FINAL]["weights"])
    model.crop_head.load_state_dict(final_heads["crop_head"].state_dict())
    model.disease_head.load_state_dict(final_heads["disease_head"].state_dict())
    model.cpu()
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    atomic_save(FULL_CHECKPOINT, {"state_dict": model.state_dict(), "settings": settings,
                                  "recipe": FINAL, "selection": selection[FINAL], "gamma": gamma[FINAL],
                                  "seen_pairs": train_pairs.tolist()})
    atomic_save(ABLATION_HEADS, {name: result["weights"] for name, result in best.items() if name != FINAL})
    saved = {"settings": settings, "selection": selection, "gamma": gamma,
             "seen_pairs": train_pairs.tolist(),
             "calibration_holdout_rows": torch.where(~fit_mask)[0].tolist(),
             "primary_recipe": FINAL,
             "full_checkpoint_bytes": FULL_CHECKPOINT.stat().st_size,
             "full_checkpoint_sha256": sha256_file(FULL_CHECKPOINT),
             "ablation_heads_sha256": sha256_file(ABLATION_HEADS)}
    pd.DataFrame([{"recipe": name, **record, "raw_gamma": 0.0, "locked_calibrated_gamma": gamma.get(name)}
                  for name, record in selection.items()]).to_csv(FINAL_DIR / "selected_models.csv", index=False)
    atomic_json(LOCK, saved)
    return saved


run = train_and_save()
display(pd.DataFrame(run["selection"]).T)

## 5. Official evaluation

Formal predictions use the locked checkpoints and 333 candidates. The ten official unseen identities are introduced here, and per-image official truth is read after score tables exist. The main experiment reports four raw rows. Two extra rows apply independently tuned Baseline 2 and A3 calibrations. Record them as diagnostic comparisons pending lecturer approval.

Use the official comparison for the common 333-candidate ranking. The supplemental restricted table uses the 323 seen and 10 unseen candidates separately, with the candidate counts stated. The three stages remain separate in the output folders.

In [ ]:
def official_vocabulary():
    # Project only concept columns. Do not load image-to-label mappings yet.
    vocabulary = pd.read_csv(team.TEST_UNSEEN_CSV, header=None, usecols=[1, 2, 3],
                             dtype=str, keep_default_na=False)
    vocabulary.columns = ["crop", "disease", "order"]
    if not vocabulary.empty and not vocabulary.iloc[0]["crop"].strip().isdigit():
        vocabulary = vocabulary.iloc[1:]
    vocabulary = vocabulary.astype("int64").drop_duplicates(["crop", "disease"]).sort_values("order")
    extra_pairs = torch.as_tensor(vocabulary[["crop", "disease"]].to_numpy(copy=True), dtype=torch.long)
    pairs = torch.cat((train_pairs, extra_pairs))
    if (len(extra_pairs) != team.NUM_UNSEEN_PAIRS or len(pairs.unique(dim=0)) != team.NUM_PAIRS
            or not set(extra_pairs[:, 0].tolist()) <= set(train_pairs[:, 0].tolist())
            or not set(extra_pairs[:, 1].tolist()) <= set(train_pairs[:, 1].tolist())):
        raise ValueError("Expected distinct official unseen pairs composed of training-derived concepts.")
    return pairs


def official_filenames(path):
    values = pd.read_csv(path, header=None, usecols=[0], dtype=str, keep_default_na=False).iloc[:, 0].tolist()
    header = {"image", "image_path", "image_name", "filename", "file_name", "file_path", "path"}
    if values and (values[0].strip().lower().replace(" ", "_") in header or
                   (len(values) > 1 and not Path(values[0]).suffix and Path(values[1]).suffix)):
        values = values[1:]
    names = [image_key(name) for name in values]
    if not names or len(set(names)) != len(names):
        raise ValueError(f"Expected unique image names in {path.name}.")
    return names


class OfficialImages(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        with Image.open(team.IMAGE_DIR / self.names[index]) as image:
            return image_transform(image.convert("RGB"))


def official_targets(csv_path, filenames, seen):
    frame = load_split(csv_path, final_evaluation=True)
    if frame.image.map(image_key).tolist() != filenames:
        raise ValueError("Official targets and predictions have different image orders.")
    labels = torch.as_tensor(frame.pair_row.to_numpy(copy=True), dtype=torch.long)
    if not ((labels < team.NUM_SEEN_PAIRS) == seen).all():
        raise ValueError("Official test split does not match seen membership.")
    return labels


@torch.inference_mode()
def predict_official(backbone, heads, pairs, filenames, title, perturb=None):
    loader = DataLoader(OfficialImages(filenames), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    scores = {recipe: [] for recipe in heads}
    mean, std = (torch.tensor(value).view(1, 3, 1, 1) for value in team.NORM_STATS["imagenet"])
    for step, images in enumerate(loader, 1):
        if perturb is not None:
            # Match the shared Baseline 2 robustness test in pixel space.
            images = (perturb(images * std + mean) - mean) / std
        features = backbone(images.to(DEVICE))
        for recipe, unit in heads.items():
            batch = pair_scores(*forward_heads(unit, features), pairs).cpu()
            if batch.shape != (len(images), len(pairs)) or not torch.isfinite(batch).all():
                raise ValueError("Unexpected candidate predictions.")
            scores[recipe].append(batch)
        if step == 1 or step == len(loader) or step % 50 == 0:
            print(f"{title}: batch {step}/{len(loader)}", flush=True)
    return {recipe: torch.cat(parts) for recipe, parts in scores.items()}


def make_rules(raw, membership, gamma_by_recipe):
    rules = {(name, "raw"): scores for name, scores in raw.items()}
    for recipe, gamma in gamma_by_recipe.items():
        rules[(recipe, "calibrated")] = raw[recipe] - gamma * membership
    return rules


def top_predictions(scores, offset=0):
    return {k: scores.topk(k, dim=1).indices + offset for k in (1, 5)}


def prediction_accuracy(predictions, labels):
    return round(100 * predictions.eq(labels[:, None]).any(1).float().mean().item(), 2)


def official_evaluation():
    if DEBUG:
        raise RuntimeError("Official evaluation requires DEBUG=False.")
    if read_final_lock() != run:
        raise RuntimeError("The in-memory selection differs from the saved final lock.")
    OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
    report_path = OFFICIAL_DIR / "official_report.json"
    if report_path.is_file():
        existing = json.loads(report_path.read_text(encoding="utf-8"))
        if existing["lock"] != run:
            raise RuntimeError("Previous evaluation belongs to another checkpoint.")
        return existing
    saved = torch.load(FULL_CHECKPOINT, map_location="cpu", weights_only=True)
    model = MultiTaskNet(pretrained=False)
    model.load_state_dict(saved["state_dict"])
    model.to(DEVICE).eval()
    heads = {FINAL: trainable_heads(model).eval()}
    for recipe, weights in torch.load(ABLATION_HEADS, map_location="cpu", weights_only=True).items():
        heads[recipe] = trainable_heads(model).eval()
        heads[recipe].load_state_dict(weights)
    heads = {recipe: heads[recipe] for recipe in RECIPES}
    if saved["seen_pairs"] != train_pairs.tolist() or run["seen_pairs"] != train_pairs.tolist():
        raise ValueError("Training candidate ordering changed after selection.")

    pairs = official_vocabulary()
    paths = (team.TEST_SEEN_CSV, team.TEST_UNSEEN_CSV)
    filenames = tuple(official_filenames(path) for path in paths)
    if (set(filenames[0]) & set(filenames[1])
            or set(train_full.image) & (set(filenames[0]) | set(filenames[1]))):
        raise ValueError("Train and official image filenames overlap.")
    membership = torch.arange(len(pairs)) < len(train_pairs)
    raw_seen = predict_official(model.backbone, heads, pairs, filenames[0], "seen")
    raw_unseen = predict_official(model.backbone, heads, pairs, filenames[1], "unseen")
    seen = make_rules(raw_seen, membership, run["gamma"])
    unseen = make_rules(raw_unseen, membership, run["gamma"])
    perturbed = {}
    robust_heads = {name: heads[name] for name in (BASELINE, FINAL)}
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(team.SEED)
        for name, transform in PERTURBATIONS.items():
            p = predict_official(model.backbone, robust_heads, pairs, filenames[0], name, transform)
            perturbed[name] = make_rules(p, membership, run["gamma"])

    boundary = len(train_pairs)
    joint_predictions = {key: (top_predictions(seen[key]), top_predictions(unseen[key])) for key in seen}
    restricted_predictions = {key: (top_predictions(seen[key][:, :boundary]),
                                    top_predictions(unseen[key][:, boundary:], offset=boundary))
                              for key in seen}
    robust_predictions = {name: {key: scores.topk(1, dim=1).indices for key, scores in variants.items()}
                          for name, variants in perturbed.items()}

    # Every raw, calibrated, restricted and robustness prediction is frozen before ground truth access.
    if not torch.equal(pairs, get_pairs(final_evaluation=True)):
        raise ValueError("Official vocabulary order differs from the shared API.")
    y_seen = official_targets(paths[0], filenames[0], seen=True)
    y_unseen = official_targets(paths[1], filenames[1], seen=False)
    y_all = torch.cat((y_seen, y_unseen))
    records, restricted, robustness = [], [], []
    for key, (seen_pred, unseen_pred) in joint_predictions.items():
        recipe, scoring = key
        row = {"recipe": recipe, "scoring": scoring, "gamma": run["gamma"][recipe] if scoring == "calibrated" else 0.0,
               "candidates": len(pairs), "epoch": run["selection"][recipe]["epoch"]}
        for k in (1, 5):
            row[f"seen_top{k}"] = prediction_accuracy(seen_pred[k], y_seen)
            row[f"unseen_top{k}"] = prediction_accuracy(unseen_pred[k], y_unseen)
            row[f"overall_top{k}"] = prediction_accuracy(torch.cat((seen_pred[k], unseen_pred[k])), y_all)
        row["harmonic_top1"] = harmonic_mean(row["seen_top1"], row["unseen_top1"])
        records.append(row)
        restricted_seen, restricted_unseen = restricted_predictions[key]
        for split, pred, target, count in (("seen_only", restricted_seen, y_seen, boundary),
                                           ("unseen_only", restricted_unseen, y_unseen, len(pairs) - boundary)):
            restricted.append({"recipe": recipe, "scoring": scoring, "split": split, "candidates": count,
                               **{f"top{k}": prediction_accuracy(pred[k], target) for k in (1, 5)}})
        if recipe in robust_heads:
            for name, variants in robust_predictions.items():
                value = prediction_accuracy(variants[key], y_seen)
                robustness.append({"recipe": recipe, "scoring": scoring, "condition": name,
                                   "seen_top1": value, "drop_pp": round(row["seen_top1"] - value, 2)})
            stem = (OFFICIAL_DIR / f"{recipe}_{scoring}").relative_to(team.RESULTS_DIR).as_posix()
            save_confusion(seen[key], y_seen, stem, final_evaluation=True)
    pd.DataFrame(records).to_csv(OFFICIAL_DIR / "official_comparison.csv", index=False)
    pd.DataFrame(restricted).to_csv(OFFICIAL_DIR / "restricted_candidates.csv", index=False)
    pd.DataFrame(robustness).to_csv(OFFICIAL_DIR / "official_robustness.csv", index=False)
    report = {"lock": run, "test_counts": {"seen": len(y_seen), "unseen": len(y_unseen)},
              "comparison": records, "restricted_candidates": restricted, "robustness": robustness}
    atomic_json(report_path, report)
    return report


if DEBUG:
    print("Debug complete. Set DEBUG=False for complete training, model export and official evaluation.")
else:
    report = official_evaluation()
    display(pd.DataFrame(report["comparison"]))
    print(f"Full Approach 2 checkpoint: {FULL_CHECKPOINT} ({FULL_CHECKPOINT.stat().st_size / 1e6:.1f} MB)")
print(f"Development results: {DEVELOPMENT_DIR.resolve()}")
if not DEBUG:
    print(f"Final selection: {FINAL_DIR.resolve()}")
    print(f"Official reports: {OFFICIAL_DIR.resolve()}")

## 6. Save

| Stage | Outputs |
|---|---|
| `results/approach2_baseline2_matched/development/` | Training history, train-only calibration grid and held-out check. |
| `results/approach2_baseline2_matched/final/` | Selected epochs, validation results and the locked scoring rule and checkpoint hashes. Model selection uses the same seen validation as Baseline 2. |
| `results/approach2_baseline2_matched/official/` | Joint and restricted candidate metrics, confusion matrices, robustness metrics and the final JSON report. |
| `checkpoints/approach2_baseline2_matched/` | Full FP32 `model.pt` and a small `comparison_heads.pt` for the other ablations. |

The final stage freezes the Baseline 2-style selected checkpoint. The complete model includes the frozen DINOv2 backbone and the A3 classifier heads, about 344 MB. Reload with `MultiTaskNet(pretrained=False)` and `model.load_state_dict(checkpoint["state_dict"])`. `DEBUG=True` only saves development logs in its debug folder.